# 리샘플링 오차 표본 수정 정리 (2026-10-08)


## 1. 문제

리샘플링 구간은 과거 예측 오차를 모아 현재 점예측에 더해 만든다. 기존 코드는 origin이 끝날 때마다 **h=1~4 오차를 전부** 풀에 넣었다.

- origin o의 h단계 오차는 분기 `o + h - 1`의 실제값이 있어야 계산된다.
- origin o+1 시점에는 분기 `o`까지만 관측돼 있다. 그래서 o의 h=2,3,4 오차(정답이 분기 o+1, o+2, o+3)는 **아직 모르는 실제값**을 쓴 것이다.

예: origin 36(2019Q4까지 관측)에서 h=2 풀에 origin 35의 h=2 오차(정답 2020Q1)가 들어 있다. 2020Q1은 origin 36 시점에 아직 관측되지 않았다.

- 풀은 h마다 따로 있어서 **채점 중인 바로 그 분기의 오차가 들어가지는 않는다.** 들어가는 것은 바로 인접한 분기의 결과(힌트)다.
- 연속된 분기의 오차는 서로 비슷한 경향(급등 직후 오차가 큼 등)이 있어, 이 힌트가 구간을 약간 유리하게 만든다.
- h=1 오차는 항상 확정된 값이라 영향이 없다.
- 영향받는 것은 리샘플링 구간뿐이다. 점예측, MSE, 자체(정규) 구간은 영향이 없다.


## 2. 수정 규칙

예측 시점 o, 단계 h의 풀에는 **`o' + h <= o`인 오차만** 쓴다(실제값이 o 이전에 확정된 것).

- 첫 채점 origin 35의 풀 크기: h=1 15개, h=2 14개, h=3 13개, h=4 12개(기존은 모두 15개).
- 그래서 수정 효과는 "미래 정보 제거"와 "초기 풀이 작아짐"이 섞여 있고, 둘을 따로 분리하지는 않았다.

**예시: 예측 시점 2020Q1(origin 36)** -- 이 시점에 관측된 마지막 분기는 2019Q4이고, 정답 분기가 2020Q1 이후인 오차가 미래값이다.

| 단계 | 기존에 쓴 origin | 미래값이 들어간 수 | 수정 후 쓰는 origin | 미래값이 들어간 수 |
|---|---|---|---|---|
| h=1 | 2016Q1 ~ 2019Q4 (16개) | 0 | 2016Q1 ~ 2019Q4 (16개) | 0 |
| h=2 | 2016Q1 ~ 2019Q4 (16개) | 1 (2019Q4) | 2016Q1 ~ 2019Q3 (15개) | 0 |
| h=3 | 2016Q1 ~ 2019Q4 (16개) | 2 (2019Q3, 2019Q4) | 2016Q1 ~ 2019Q2 (14개) | 0 |
| h=4 | 2016Q1 ~ 2019Q4 (16개) | 3 (2019Q2, 2019Q3, 2019Q4) | 2016Q1 ~ 2019Q1 (13개) | 0 |

기존에는 h와 상관없이 h=1 기준(직전 origin까지)의 같은 origin을 모두 써서, h가 클수록 미래값이 1개씩 더 들어갔다.

## 3. 수정 범위

| 구분 | 상태 |
|---|---|
| 11시리즈 (11-1~11-4) | 수정 완료. 11-1이 확정된 오차만으로 리샘플링을 다시 계산하고 `cache_107_rank_compare_df.pkl`에 저장. 11-2~11-4는 이 캐시만 읽음 |
| 9시리즈, 10시리즈, PPT 9·10번 | **미수정**(기존 숫자 그대로, 같은 문제 있음) |
| 8-1, 6-1 등 이전 시리즈 | 같은 구조임을 코드로 확인(8-1, 6-1). `rng.choice`로 과거 오차를 뽑는 노트북이 30개 있고 나머지는 하나씩 확인하지 않음. 미수정 |


## 4. 영향 (리샘플링 8개 조합, 수정 전 -> 수정 후)

| 모델 | 스케일 | CRPS(`crps_ensemble`) | 변화 | 커버리지(%) | 폭 |
|---|---|---|---|---|---|
| SV | 원 | 0.5842 -> 0.5987 | +2.5% | 81.8 -> 83.0 | 3.48 -> 3.48 |
| AR단독 | 원 | 0.5868 -> 0.5986 | +2.0% | 69.3 -> 63.6 | 2.20 -> 2.10 |
| auto_arima | 원 | 0.5904 -> 0.5955 | +0.9% | 69.3 -> 69.3 | 2.23 -> 2.14 |
| GARCH | 원 | 0.6197 -> 0.6248 | +0.8% | 84.1 -> 84.1 | 5.05 -> 5.01 |
| SV | 로그 | 0.7010 -> 0.7146 | +1.9% | 85.2 -> 87.5 | 7.85 -> 8.05 |
| AR단독 | 로그 | 0.7250 -> 0.7250 | 0.0% | 78.4 -> 70.5 | 5.73 -> 5.35 |
| auto_arima | 로그 | 0.8151 -> 0.8064 | -1.1% | 77.3 -> 71.6 | 6.16 -> 5.71 |
| GARCH | 로그 | 0.9668 -> 0.9723 | +0.6% | 89.8 -> 89.8 | 15.55 -> 19.59 |

- 원 스케일 리샘플링 CRPS는 모두 나빠졌다(+0.8~2.5%). 로그 스케일은 방향이 섞여 있다.
- GARCH 로그 리샘플링은 폭이 15.6에서 19.6으로 오히려 넓어졌다.
- 영향은 작지만(대부분 3% 이내) 상위 후보들의 CRPS 차이(약 2.6%)와 비슷한 크기라 선정에 영향을 줄 수 있다.


## 5. 선정에 미친 영향 (삭제한 이전 11-5, 11-6의 기록)

- 이전 11-5(k를 88개 전체로 정한 단계식 선정): SV 원 리샘플링 -> **SV 원 자체**로 바뀜. 이유는 SV 원 리샘플링의 CRPS가 2% 이내 범위(CRPS 1등 0.5783 대비 +1.0% -> +3.5%)를 벗어났기 때문이다.
- 이전 11-6(walk-forward k): SV 원 리샘플링이 계속 선정되지만 CRPS가 1등보다 3.1% 높던 것이 5.5%로 커졌다. 27가지 허용 범위에서는 SV 원 자체 15번, SV 원 리샘플링 12번으로 갈렸다.
- 이전 11-5, 11-6은 삭제했고(파일 백업: scratchpad `removed_11-5_6`), 현재 11-5는 4단계 필터 비교로 새로 만든 노트북이고, 스케일 계수(k) 보정 연구는 `11-6`이다.


## 6. 확인된 것과 남은 것

확인된 것
- 점예측과 MSE, 자체 구간은 수정 전후 값이 같다(MSE 최대 차이 0, 자체 구간 crps7 차이 1e-15).
- 11시리즈 노트북이 모두 0 오류로 재실행됨.

남은 것
- 9·10시리즈와 PPT를 같은 방식으로 수정할지 결정 필요(현재는 보류).
- 보정(k) 연구(11-6)의 나머지 질문(구간 점수, k 안정성, 공통 k vs h별 k 등)을 이어서 진행.